# Module 1 · Structure and Central Canopy Cropping
**15 minutes: principles 4 · worked example 3 · transfer exercises 6 · discussion 2**

[Course introduction](00_Introduction.ipynb) · [Runnable reference answer](#reference-answer)

**Learning objectives:** distinguish leaf shape from orientation; translate a triangle-area
equation into NumPy; calculate LAI using the actual central ground area.


In [ ]:
# Run this preparation cell once after starting/restarting the kernel.
# Finds normal, flattened, nested and ZIP-preserved course uploads.
from pathlib import Path
import sys,os,subprocess,hashlib,zipfile,shutil

def locate_canopy_runtime(input_root=None,local_roots=None,work=None,tag=None):
    tag=tag or ('win_amd64' if sys.platform=='win32' else 'manylinux_2_34_x86_64')
    expected=f'canopyteach-1.0.1-cp311-abi3-{tag}.whl'
    pattern=f'canopyteach-1.0.1-cp311-abi3-{tag}*.whl'
    work=Path(work or '/kaggle/working/canopyteach_practical_en')
    candidate_root=Path(input_root or '/kaggle/input')
    wheels=set();helpers=set();archives=set();observed=[];archive_errors=[]

    def consider(p):
        if p.name=='canopy_practical.py':helpers.add(p.resolve())
        elif p.match(pattern):wheels.add(p.resolve())
        elif p.suffix.lower()=='.zip':archives.add(p.resolve())
        if len(observed)<35 and p.is_file():observed.append(str(p))

    if candidate_root.is_dir():
        # Kaggle can add dataset/owner/version directories or flatten uploads.
        for p in candidate_root.rglob('*'):
            if p.is_file():consider(p)
    else:
        roots=local_roots if local_roots is not None else ([] if input_root is not None else [Path.cwd(),*Path.cwd().parents])
        for folder in map(Path,roots):
            if folder==Path(folder.anchor):continue
            for p in [folder/'canopy_practical.py',*folder.glob(pattern),*folder.glob('wheels/'+pattern)]:
                if p.is_file():consider(p)
        if input_root is not None:
            observed.append('Input directory does not exist: '+str(candidate_root))

    # Read only the runtime members of course ZIPs into a writable directory.
    # Never extract arbitrary archive paths or change the input Dataset.
    if not wheels or not helpers:
        for archive in sorted(archives):
            try:
                with zipfile.ZipFile(archive) as z:
                    selected=[n for n in z.namelist() if not n.endswith('/') and
                        (Path(n).name=='canopy_practical.py' or Path(n).match(pattern))]
                    if not selected:continue
                    key=hashlib.sha256(str(archive).encode()).hexdigest()[:12]
                    for i,name in enumerate(selected):
                        info=z.getinfo(name)
                        if info.file_size>30_000_000:raise ValueError('Runtime member exceeds 30 MB: '+name)
                        dest=work/'_runtime'/key/str(i)/Path(name).name
                        dest.parent.mkdir(parents=True,exist_ok=True)
                        dest.write_bytes(z.read(name));consider(dest)
            except (zipfile.BadZipFile,OSError,ValueError) as exc:
                archive_errors.append(str(archive)+': '+str(exc))

    if not wheels or not helpers:
        missing=[]
        if not wheels:missing.append(expected)
        if not helpers:missing.append('canopy_practical.py')
        details='\n'.join(observed) or '(no mounted input files found)'
        raise FileNotFoundError(
            'Missing course runtime: '+', '.join(missing)+'\n'
            'Add the course Dataset as an input to THIS notebook; importing an .ipynb alone does not attach its Dataset.\n'
            'Searched input: '+str(candidate_root)+'\nObserved files (first 35):\n'+details+
            ('\nArchive errors:\n'+'\n'.join(archive_errors) if archive_errors else ''))

    # Prefer a nearby helper, without requiring the wheels/ directory layout.
    pairs=[]
    for wheel in sorted(wheels):
        for helper in sorted(helpers):
            shared=len(os.path.commonpath([str(wheel.parent),str(helper.parent)]).split(os.sep))
            nearby=helper.parent in (wheel.parent,wheel.parent.parent)
            pairs.append((not nearby,-shared,str(wheel),str(helper)))
    _,_,wheel_name,helper_name=min(pairs)
    wheel,helper=Path(wheel_name),Path(helper_name)
    # Browser downloads may add a suffix such as " (1)" to a wheel filename.
    if wheel.name!=expected:
        normalized=work/'_runtime'/'normalized'/expected
        normalized.parent.mkdir(parents=True,exist_ok=True)
        shutil.copyfile(wheel,normalized);wheel=normalized
    return wheel,helper

if __name__=='__main__':
    if sys.version_info<(3,11):raise RuntimeError('This course requires CPython 3.11 or newer.')
    input_override=os.environ.get('CANOPYTEACH_INPUT_ROOT')
    work_override=os.environ.get('CANOPYTEACH_WORK_ROOT')
    on_kaggle=Path('/kaggle/working').is_dir()
    WORK=Path(work_override) if work_override else (
        Path('/kaggle/working/canopyteach_practical_en') if on_kaggle else Path.cwd()/'results/practical_en')
    wheel,helper=locate_canopy_runtime(input_root=input_override,work=WORK)
    PACKAGE_ROOT=helper.parent
    WORK.mkdir(parents=True,exist_ok=True);site=WORK/'_packages';marker=site/'course_wheel.sha256'
    signature=hashlib.sha256(wheel.read_bytes()).hexdigest()
    if not marker.exists() or marker.read_text().strip()!=signature:
        if 'canopyteach' in sys.modules:
            raise RuntimeError('Restart the kernel before installing a different course wheel, then run preparation again.')
        subprocess.run([sys.executable,'-m','pip','install','--no-index','--no-deps','--disable-pip-version-check',
                        '--upgrade','--target',str(site),str(wheel)],check=True)
        marker.write_text(signature)
    sys.path.insert(0,str(site));sys.path.insert(0,str(PACKAGE_ROOT))
    import canopy_practical as cp
    import canopyteach
    if canopyteach.__version__!='1.0.1':raise RuntimeError('Unexpected canopyteach version; restart the kernel.')
    cp.style()
    try:get_ipython().run_line_magic('matplotlib','inline')
    except NameError:pass
    DEFAULT_STEP=int(os.environ.get('CANOPYTEACH_STEP_MINUTES','30'))
    DEFAULT_SPACING=float(os.environ.get('CANOPYTEACH_RAY_SPACING','.005'))
    print('Runtime wheel:',wheel)
    print('Teaching helper:',helper)
    print('Ready. Outputs:',WORK)
    print('Random whole-plant azimuth; central crop; XY periodic; C/OpenMP; synthetic direct light.')


## 1. Principles · 4 min

A tapered stem has radius $r(z)=r_0(1-kz/H)$.
The supplied maize model preserves its width/PCHIP profile, curved midrib and edge waves.
Leaf inclination is measured above horizontal. Changing only inclination rotates a fixed
leaf shape rigidly and preserves its surface area.

For triangle vertices $p_0,p_1,p_2$,
$$A=\frac12\|(p_1-p_0)\times(p_2-p_0)\|.$$
Central ground area is $A_g=(n_xd_x)(n_yd_y)$ and $LAI=\sum_{leaf} A_i/A_g$.
LAI is dimensionless.

Spatially clip a 3 x 3 central cell from a 7 x 7 stand. Keep neighboring leaf fragments
entering the cell; selecting only the nine central plant IDs would omit them.
The actual boundary-triangle clipping is provided.

**Whole-plant azimuth:** each plant has one seeded uniform 0–360° rotation about Z.
Its original 180° within-plant leaf sequence is retained. The same central-cell angle map
is repeated into the source buffer and across periodic boundaries. Seed 20261025 makes
comparisons reproducible; keep that realization fixed when comparing leaf angles.
The former all-aligned canopy is retained only as a labelled diagnostic control.


## 2. Worked example · 3 min

Run the supplied stem-radius function: middle/top radii should be **6.4/4.8 mm**.
Set `leaf_length`, `leaf_width` and `leaf_angle`, then rerun the whole cell.
Four panels compare a reference leaf with changes in length only, width only and angle only.
The supplied maize width profile, natural midrib curvature and edge waves are retained.

Length is **midrib arc length**, not an X-axis bounding-box length. Width is a profile input;
rotation changes the XYZ spans. Angle changes preserve actual area; length/width changes can
change area. Read the area table rather than inferring area from the view.
All four panels use identical XYZ limits, equal metre scales and an orthographic view.
The full negative-Z part of an isolated curved leaf is shown; model vertices are never rescaled.

The canopy view displays the central cell. **XY-periodic rays are already enabled**:
a ray leaving one side is translated to the opposite side and continues, accounting for
repeated neighboring canopy. The drawing is not an open-boundary light calculation.

This example is complete. In the next cell implement triangle surface area.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import canopy_practical as cp
cp.style()

leaf_length=.80  # EDIT: midrib arc length in metres.
leaf_width=.16   # EDIT: width-profile input in metres.
leaf_angle=60.   # EDIT: elevation above horizontal, in degrees.
def supplied_stem_radius(z,H=1.6,r0=.008,taper=.4):
    return r0*(1-taper*z/H)
print('Stem middle/top radius (mm):',1000*supplied_stem_radius(.8),1000*supplied_stem_radius(1.6))
# Four panels change one factor at a time; the original curved profile is retained.
fig=cp.geometry_example(angle=leaf_angle,length=leaf_length,width=leaf_width);plt.show()
fig=cp.crop_figure();plt.show()
_,info=cp.geometry_trial();cp.print_geometry(info)


## 3. Transfer programming exercise 1A–1B · 4 min

**1A · Write the area expression.** Complete the return statement in
`student_triangle_area`. `np.cross(u,v)` computes the cross product;
`np.linalg.norm(..., axis=-1)` takes its length along the coordinate axis.
Your function must handle one triangle and an array of triangles.
The right-triangle self-check should give **0.5 m2**.

**1B · Calculate LAI.** Replace `student_lai=None` with leaf area divided by central
ground area. Select leaf faces with `mesh.leaf_mask` before summing `areas`.
Reference central ground area: **1.584 m2**; LAI: approximately **2.807287**.
The area of the entire 7 x 7 stand is not the correct denominator.


In [ ]:
import numpy as np
import canopy_practical as cp

def student_triangle_area(p0,p1,p2):
    u=np.asarray(p1)-np.asarray(p0)
    v=np.asarray(p2)-np.asarray(p0)
    return None  # TODO 1A: half the length of cross(u,v); use axis=-1.

probe=student_triangle_area([0,0,0],[1,0,0],[0,1,0])
if probe is None:
    print('1A is unfinished: complete the area return expression. The check should give 0.5 m2.')
else:
    np.testing.assert_allclose(probe,.5)
    mesh,info=cp.geometry_trial()
    t=mesh.triangles
    areas=student_triangle_area(t[:,0],t[:,1],t[:,2])
    np.testing.assert_allclose(areas,mesh.area,rtol=1e-9,atol=1e-12)
    student_lai=None  # TODO 1B: leaf area sum / central ground area.
    if student_lai is None:
        print('1A passed. Complete student_lai; mesh.leaf_mask selects the leaf faces.')
    else:
        np.testing.assert_allclose(student_lai,mesh.lai)
        print('1A–1B complete. LAI =',student_lai)


## 4. Independent structure experiment 1C · 2 min

Change `canopy_leaf_length`, `canopy_leaf_width`, `lower_angle` and `upper_angle`, then rerun
the whole cell. Start with one factor at a time. The comparison panels share the same cubic
XYZ scale. Length/width changes may change actual area and LAI; the output explicitly says
whether area is controlled. Return dimensions to **0.65 / 0.12 m** for an angle-only comparison.

This remains runnable while the area exercise is unfinished. Record the number of central
stems and of plants contributing leaves. Why are these counts different?
Module 3 keeps dimensions fixed so that its design contrast isolates leaf angles.


In [ ]:
from dataclasses import replace
import numpy as np
import matplotlib.pyplot as plt
from canopyteach import Architecture
import canopy_practical as cp
cp.style()

canopy_leaf_length=.65  # EDIT: metres; changing length changes leaf area/LAI.
canopy_leaf_width=.12   # EDIT: metres; changing width changes leaf area/LAI.
lower_angle=15.        # EDIT: degrees above horizontal.
upper_angle=75.        # EDIT: degrees above horizontal.
base=Architecture()
selected=replace(base,leaf_length=canopy_leaf_length,leaf_width=canopy_leaf_width,
                 lower_angle=lower_angle,upper_angle=upper_angle)
fig=cp.structures({'baseline':base,'my_structure':selected});plt.show()
baseline,info0=cp.geometry_trial()
mesh,info=cp.geometry_trial(lower_angle,upper_angle,length=canopy_leaf_length,width=canopy_leaf_width)
cp.print_geometry(info)
print('Leaf area difference (m2):',info['leaf_area_m2']-info0['leaf_area_m2'])
if canopy_leaf_length==base.leaf_length and canopy_leaf_width==base.leaf_width:
    np.testing.assert_allclose(info['LAI'],info0['LAI'],rtol=1e-10)
    print('Angle-only comparison: actual leaf area and LAI are controlled.')
else:
    print('Dimension experiment: LAI can change. Do not attribute the difference to angle alone.')
# No unfinished student function is needed. Invalid below-ground shapes raise a clear error.


<a id="reference-answer"></a>
## 5. Reference answers and discussion · 2 min

**1A:** half the length of the cross product. **1B:** sum only leaf area and divide by 1.584.
Rigid inclination changes preserve area; changes in curvature or width need not do so.

The default central cell has **9 stems**, but leaf parts from **19 plants** enter it.
Selecting nine whole plants would omit incoming leaves. Wrapping all 49 complete plants
into the center would instead duplicate area.

The local reference function below respects the selected angles, or uses 15/75 degrees
in a fresh kernel. It does not replace your function or parameter choices.
Write two sentences identifying the expressions you implemented and whether LAI was controlled.
Continue to [Module 2](02_Light_and_Daily_Integration.ipynb).


In [ ]:
def _reference_geometry(selected_lower,selected_upper,length=.65,width=.12):
    import numpy as np
    import canopy_practical as cp
    def area(p0,p1,p2):
        u=np.asarray(p1)-np.asarray(p0);v=np.asarray(p2)-np.asarray(p0)
        return .5*np.linalg.norm(np.cross(u,v),axis=-1)
    np.testing.assert_allclose(area([0,0,0],[1,0,0],[0,1,0]),.5)
    mesh,info=cp.geometry_trial(selected_lower,selected_upper,length=length,width=width)
    t=mesh.triangles;areas=area(t[:,0],t[:,1],t[:,2])
    lai=areas[mesh.leaf_mask].sum()/mesh.ground_area
    np.testing.assert_allclose(lai,mesh.lai)
    print('Area function verified; LAI:',lai);cp.print_geometry(info)
_reference_geometry(globals().get('lower_angle',15.),globals().get('upper_angle',75.),
                    globals().get('canopy_leaf_length',.65),globals().get('canopy_leaf_width',.12))
